In [ ]:
# full_improved_isles_with_sampler_and_focaltversky.py
# Ensemble training with caching, RAM preload, sequential per-model training, shape fixes,
# WeightedRandomSampler to oversample lesion-containing cases, FocalTversky loss, and
# validation tuning for threshold and min-component filtering.
#
# Set SEARCH_ROOT to your dataset root before running.

import sys
import subprocess
import os
import glob
import random
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

def pip_install(packages):
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade"] + packages
    subprocess.check_call(cmd)

# --- Ensure required packages (use correct names) ---
required = []
try:
    import monai  # noqa: F401
except Exception:
    required.append("monai")

try:
    import sklearn  # noqa: F401
except Exception:
    required.append("scikit-learn")

try:
    import SimpleITK  # noqa: F401
except Exception:
    required.append("SimpleITK")

if required:
    print("Installing missing packages:", required)
    pip_install(required)

# --- Standard imports (after install) ---
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
from scipy.ndimage import zoom, label, binary_closing
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler
from torch.cuda.amp import GradScaler

from monai.networks.nets import DynUNet, AttentionUnet, UNet, SegResNet
from monai.losses import DiceLoss, TverskyLoss
from monai.transforms import Compose, RandFlipd, RandGaussianNoised
from monai.metrics import DiceMetric

# Speed / backend tweaks
torch.backends.cudnn.benchmark = True

# ---------------- Config ----------------
SEARCH_ROOT = "/kaggle/input/"  # <-- change to your dataset root
OUT_DIR = "./checkpoints_improved"
CACHE_DIR = "./preprocessed_cache"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

CONFIG = {
    "in_channels": 3,
    "out_channels": 1,
    "lr": 1e-4,
    "epochs": 100,
    "batch_size": 1,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    # MUST be product of downsample strides (for models with 4 downsamples -> 2^4 = 16)
    "spatial_multiple": 16,
    "seed": 42,
    # patch_size dimensions must be divisible by spatial_multiple (16)
    "patch_size": (64, 128, 128),
    "lesion_patch_prob": 0.9,  # increased to show more lesion patches
    "min_component_voxels": 50,
    "tta_transforms": ["flip_x", "flip_y", "flip_z"],
    "num_workers_train": min(8, max(1, cpu_count()//2)),
    "num_workers_val": min(4, max(1, cpu_count()//4)),
    "validate_every": 1,
    "compile_models": False,
    "preload_to_ram": True,
    "apply_n4_in_preprocessing": False,
    "prefetch_factor": 2
}
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])

# ---------------- Utilities ----------------
def find_real_nii_file(path_pattern: str):
    matches = glob.glob(path_pattern, recursive=True)
    for path in matches:
        if os.path.isdir(path):
            nii_files = glob.glob(os.path.join(path, "**", "*.nii*"), recursive=True)
            if nii_files:
                return nii_files[0]
        elif path.endswith(".nii") or path.endswith(".nii.gz"):
            return path
    return None

def n4_bias_correction(numpy_img):
    try:
        img = sitk.GetImageFromArray(numpy_img.astype(np.float32))
        maskImage = sitk.OtsuThreshold(img, 0, 1, 200)
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        out = corrector.Execute(img, maskImage)
        out_np = sitk.GetArrayFromImage(out).astype(np.float32)
        return out_np
    except Exception:
        return numpy_img

def zscore_normalize(volume):
    v = volume.astype(np.float32)
    m = v.mean()
    s = v.std() if v.std() > 0 else 1.0
    return (v - m) / s

def pad_to_multiple_tensor(x: torch.Tensor, multiple: int):
    _, _, D, H, W = x.shape
    pad_d = (multiple - (D % multiple)) % multiple
    pad_h = (multiple - (H % multiple)) % multiple
    pad_w = (multiple - (W % multiple)) % multiple
    pad = (0, pad_w, 0, pad_h, 0, pad_d)
    if any([pad_d, pad_h, pad_w]):
        x = F.pad(x, pad, mode="constant", value=0)
    return x, pad

def unpad_tensor(x: torch.Tensor, pad):
    if pad is None:
        return x
    w_r = pad[1]; h_r = pad[3]; d_r = pad[5]
    D, H, W = x.shape[2], x.shape[3], x.shape[4]
    d_end = D - d_r if d_r != 0 else D
    h_end = H - h_r if h_r != 0 else H
    w_end = W - w_r if w_r != 0 else W
    return x[:, :, :d_end, :h_end, :w_end]

def pad_volume_to_multiple(vol, multiple):
    D, H, W = vol.shape
    pad_d = (multiple - (D % multiple)) % multiple
    pad_h = (multiple - (H % multiple)) % multiple
    pad_w = (multiple - (W % multiple)) % multiple
    if pad_d or pad_h or pad_w:
        vol = np.pad(vol, ((0,pad_d),(0,pad_h),(0,pad_w)), mode="constant", constant_values=0)
    return vol

def remove_small_components(pred_mask_np, min_voxels=50):
    labeled, n = label(pred_mask_np)
    out = np.zeros_like(pred_mask_np)
    for comp in range(1, n+1):
        comp_vox = (labeled == comp)
        if comp_vox.sum() >= min_voxels:
            out[comp_vox] = 1
    return out

# ---------------- Data pairing ----------------
def get_isles_dataframe(search_path: str):
    mask_files = glob.glob(os.path.join(search_path, "**", "*_msk.nii*"), recursive=True)
    data = []
    for mask_path in tqdm(mask_files, desc="Pairing modalities"):
        filename = os.path.basename(mask_path)
        parts = filename.split("_")
        if len(parts) < 2:
            continue
        case_id = parts[0]
        session_id = parts[1]
        if "derivatives" in mask_path:
            base_dir = mask_path.split("derivatives")[0]
        else:
            base_dir = os.path.dirname(os.path.dirname(mask_path)) + os.sep
        case_folder = os.path.join(base_dir, case_id, session_id)
        flair = find_real_nii_file(os.path.join(case_folder, "anat", "*FLAIR.nii*"))
        adc = find_real_nii_file(os.path.join(case_folder, "dwi", "*adc.nii*"))
        dwi = find_real_nii_file(os.path.join(case_folder, "dwi", "*dwi.nii*"))
        if flair and adc and dwi:
            data.append({"case_id": case_id, "flair": flair, "adc": adc, "dwi": dwi, "mask": mask_path})
    return pd.DataFrame(data)

# ---------------- Preprocessing worker ----------------
def preprocess_case_worker(args):
    idx, row, cache_path, apply_n4, multiple = args
    try:
        if os.path.exists(cache_path):
            return cache_path
        flair = nib.load(row["flair"]).get_fdata().astype(np.float32)
        adc = nib.load(row["adc"]).get_fdata().astype(np.float32)
        dwi = nib.load(row["dwi"]).get_fdata().astype(np.float32)
        mask = nib.load(row["mask"]).get_fdata().astype(np.float32)
        if apply_n4:
            flair = n4_bias_correction(flair)
            adc = n4_bias_correction(adc)
            dwi = n4_bias_correction(dwi)
        target_shape = mask.shape
        def resample(vol, target_shape, is_mask=False):
            if vol.shape == target_shape:
                return vol
            zoom_factors = [t / s for s, t in zip(vol.shape, target_shape)]
            order = 0 if is_mask else 1
            return zoom(vol, zoom_factors, order=order)
        flair = resample(flair, target_shape)
        adc = resample(adc, target_shape)
        dwi = resample(dwi, target_shape)
        mask = resample(mask, target_shape, is_mask=True)
        flair = zscore_normalize(flair)
        adc = zscore_normalize(adc)
        dwi = zscore_normalize(dwi)
        flair = pad_volume_to_multiple(flair, multiple)
        adc = pad_volume_to_multiple(adc, multiple)
        dwi = pad_volume_to_multiple(dwi, multiple)
        mask = pad_volume_to_multiple(mask, multiple)
        image = np.stack([flair, adc, dwi], axis=0).astype(np.float32)
        mask = (mask > 0.5).astype(np.float32)
        np.savez_compressed(cache_path, image=image, mask=mask)
        return cache_path
    except Exception:
        return None

# ---------------- Dataset with lesion-focused patching and RAM preload ----------------
class Stroke3DDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, patch_size=None, lesion_patch_prob=0.9,
                 apply_n4=False, augment=False, preload_to_ram=True, n_proc=4):
        self.df = dataframe.reset_index(drop=True)
        self.patch_size = patch_size
        self.lesion_patch_prob = lesion_patch_prob
        self.apply_n4 = apply_n4
        self.augment = augment
        self.preload_to_ram = preload_to_ram
        self.n_proc = n_proc
        self.aug_transforms = Compose([
            RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=0),
            RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=1),
            RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=2),
            RandGaussianNoised(keys=["image"], prob=0.1, mean=0.0, std=0.01)
        ])
        self.cache_paths = []
        for i, row in self.df.iterrows():
            case_id = row["case_id"]
            cache_name = f"{case_id}_{i}.npz"
            cache_path = os.path.join(CACHE_DIR, cache_name)
            self.cache_paths.append(cache_path)
        self._ensure_cache_parallel()
        self.memory_cache = None
        if self.preload_to_ram:
            self._preload_to_ram()

    def _ensure_cache_parallel(self):
        tasks = []
        multiple = CONFIG["spatial_multiple"]
        for i, row in self.df.iterrows():
            tasks.append((i, row, self.cache_paths[i], self.apply_n4, multiple))
        n_workers = min(self.n_proc, max(1, cpu_count()-1))
        with Pool(processes=n_workers) as p:
            for _ in tqdm(p.imap_unordered(preprocess_case_worker, tasks), total=len(tasks), desc="Preprocessing cache"):
                pass

    def _preload_to_ram(self):
        self.memory_cache = []
        for p in tqdm(self.cache_paths, desc="Loading cache to RAM"):
            data = np.load(p)
            self.memory_cache.append((data["image"], data["mask"]))

    def __len__(self):
        return len(self.df)

    def sample_patch(self, image, mask):
        D, H, W = image.shape[1:]
        pd, ph, pw = self.patch_size
        if mask.sum() > 0 and random.random() < self.lesion_patch_prob:
            coords = np.argwhere(mask > 0)
            z, y, x = coords[np.random.randint(len(coords))]
            z0 = max(0, z - pd//2); z1 = z0 + pd
            y0 = max(0, y - ph//2); y1 = y0 + ph
            x0 = max(0, x - pw//2); x1 = x0 + pw
            if z1 > D: z1 = D; z0 = max(0, D - pd)
            if y1 > H: y1 = H; y0 = max(0, H - ph)
            if x1 > W: x1 = W; x0 = max(0, W - pw)
        else:
            z0 = random.randint(0, max(0, D - pd))
            y0 = random.randint(0, max(0, H - ph))
            x0 = random.randint(0, max(0, W - pw))
            z1, y1, x1 = z0 + pd, y0 + ph, x0 + pw
        img_patch = image[:, z0:z1, y0:y1, x0:x1]
        mask_patch = mask[z0:z1, y0:y1, x0:x1]
        return img_patch, mask_patch

    def __getitem__(self, idx):
        if self.memory_cache is not None:
            image, mask = self.memory_cache[idx]
        else:
            data = np.load(self.cache_paths[idx])
            image, mask = data["image"], data["mask"]
        if self.patch_size is not None:
            image, mask = self.sample_patch(image, mask)
        mask = np.expand_dims(mask, axis=0)
        if self.augment:
            d = {"image": image, "mask": mask}
            d = self.aug_transforms(d)
            image, mask = d["image"], d["mask"]
        return torch.from_numpy(image.astype(np.float32)), torch.from_numpy(mask.astype(np.float32))

# ---------------- DerNet3D (new) ----------------
class DerNet3D(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_filters=16):
        super().__init__()
        f = base_filters
        self.enc1 = nn.Sequential(
            nn.Conv3d(in_channels, f, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(f),
            nn.ReLU(inplace=True)
        )
        self.block1 = nn.Sequential(
            nn.Conv3d(f, f, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(f),
            nn.ReLU(inplace=True)
        )
        self.down1 = nn.Sequential(
            nn.Conv3d(f*2, f*2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(f*2),
            nn.ReLU(inplace=True)
        )
        self.block2 = nn.Sequential(
            nn.Conv3d(f*2, f*2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(f*2),
            nn.ReLU(inplace=True)
        )
        self.up1 = nn.ConvTranspose3d(f*2, f, kernel_size=2, stride=2)
        self.final = nn.Conv3d(f*2, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        b1 = self.block1(e1)
        cat1 = torch.cat([e1, b1], dim=1)
        d1 = self.down1(cat1)
        b2 = self.block2(d1)
        u1 = self.up1(b2)
        if u1.shape[2:] != e1.shape[2:]:
            u1 = F.interpolate(u1, size=e1.shape[2:], mode="trilinear", align_corners=False)
        out_cat = torch.cat([u1, e1], dim=1)
        logits = self.final(out_cat)
        return logits

# ---------------- Model factory ----------------
def get_models(device):
    m1 = UNet(spatial_dims=3, in_channels=3, out_channels=1,
              channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2)).to(device)
    m2 = AttentionUnet(spatial_dims=3, in_channels=3, out_channels=1,
                       channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2)).to(device)
    m3 = SegResNet(spatial_dims=3, in_channels=3, out_channels=1, init_filters=16).to(device)
    m4 = DynUNet(
        spatial_dims=3, in_channels=3, out_channels=1,
        kernel_size=[[3,3,3]]*5, strides=[[1,1,1]] + [[2,2,2]]*4,
        upsample_kernel_size=[[2,2,2]]*4, filters=[16, 32, 64, 128, 256]
    ).to(device)
    m5 = DerNet3D(in_channels=3, out_channels=1, base_filters=16).to(device)
    models = [m1, m2, m3, m4, m5]
    if CONFIG.get("compile_models", False):
        try:
            for i, m in enumerate(models):
                models[i] = torch.compile(m)
        except Exception:
            pass
    return models

# ---------------- Focal Tversky Loss ----------------
class FocalTversky(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, target):
        probs = torch.sigmoid(logits)
        # flatten spatial dims
        dims = (2,3,4)
        TP = (probs * target).sum(dim=dims)
        FP = (probs * (1 - target)).sum(dim=dims)
        FN = ((1 - probs) * target).sum(dim=dims)
        tversky = (TP + self.smooth) / (TP + self.alpha * FN + self.beta * FP + self.smooth)
        loss = (1 - tversky) ** self.gamma
        return loss.mean()

# ---------------- Validation (no TTA) ----------------
def validate_ensemble(models, loader):
    for m in models: m.eval()
    dice_metric = DiceMetric(include_background=True, reduction="mean")
    all_f1s = []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validating", leave=False):
            images = images.to(CONFIG["device"], dtype=torch.float32)
            masks = masks.to(CONFIG["device"], dtype=torch.float32)
            if images.ndim == 4:
                images = images.unsqueeze(0); masks = masks.unsqueeze(0)
            images, pad = pad_to_multiple_tensor(images, CONFIG["spatial_multiple"])
            masks, _ = pad_to_multiple_tensor(masks, CONFIG["spatial_multiple"])
            preds = []
            for model in models:
                p = model(images)
                p = torch.sigmoid(p)
                if p.shape != masks.shape:
                    p = F.interpolate(p, size=masks.shape[2:], mode="trilinear", align_corners=False)
                preds.append(p)
            ensemble_pred = torch.mean(torch.stack(preds, dim=0), dim=0)
            final_mask = (ensemble_pred > 0.5).float()
            bin_masks = (masks > 0.5).float()
            try:
                dice_metric(y_pred=final_mask, y=bin_masks)
            except Exception:
                if final_mask.shape != bin_masks.shape:
                    final_mask = F.interpolate(final_mask, size=bin_masks.shape[2:], mode="nearest")
                    dice_metric(y_pred=final_mask, y=bin_masks)
                else:
                    raise
            final_mask_unpadded = unpad_tensor(final_mask, pad)
            bin_masks_unpadded = unpad_tensor(bin_masks, pad)
            fp = final_mask_unpadded.cpu().numpy()
            gt = bin_masks_unpadded.cpu().numpy()
            B = fp.shape[0]
            for b in range(B):
                y_pred_flat = fp[b].ravel()
                y_true_flat = gt[b].ravel()
                if y_true_flat.sum() == 0 and y_pred_flat.sum() == 0:
                    f1 = 1.0
                else:
                    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
                all_f1s.append(f1)
    mean_dice = dice_metric.aggregate().item()
    dice_metric.reset()
    mean_f1 = float(np.mean(all_f1s)) if len(all_f1s) > 0 else 0.0
    return mean_dice, mean_f1

# ---------------- Helpers for tuning threshold and min-vox ----------------
def collect_val_predictions(models, loader):
    """Return list of per-model prediction arrays and ground-truth masks (numpy)."""
    for m in models: m.eval()
    preds_per_model = [[] for _ in models]
    gts = []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Collecting val preds", leave=False):
            images = images.to(CONFIG["device"], dtype=torch.float32)
            masks = masks.to(CONFIG["device"], dtype=torch.float32)
            if images.ndim == 4:
                images = images.unsqueeze(0); masks = masks.unsqueeze(0)
            images, pad = pad_to_multiple_tensor(images, CONFIG["spatial_multiple"])
            masks, _ = pad_to_multiple_tensor(masks, CONFIG["spatial_multiple"])
            for i, model in enumerate(models):
                p = torch.sigmoid(model(images))
                if p.shape != masks.shape:
                    p = F.interpolate(p, size=masks.shape[2:], mode="trilinear", align_corners=False)
                p_unp = unpad_tensor(p, pad)
                preds_per_model[i].append(p_unp.cpu().numpy()[0,0])
            gt_unp = unpad_tensor(masks, pad)
            gts.append(gt_unp.cpu().numpy()[0,0].astype(np.uint8))
    # stack per-case
    preds_per_model = [np.stack(lst, axis=0) for lst in preds_per_model]  # shape (N_cases, D,H,W) per model
    # transpose to list of per-model arrays with shape (N_cases, D,H,W)
    return preds_per_model, gts

def tune_threshold_and_minvox(preds_per_model, gts, thresholds=np.linspace(0.3,0.7,21), min_vox_list=[10,25,50,100]):
    best = (0.5, 50, -1.0)
    # preds_per_model: list of arrays [model_idx] -> (N_cases, D,H,W)
    N = preds_per_model[0].shape[0]
    for t in thresholds:
        for mv in min_vox_list:
            f1s = []
            for i in range(N):
                ensemble = np.mean([preds_per_model[m][i] for m in range(len(preds_per_model))], axis=0)
                bin_mask = (ensemble > t).astype(np.uint8)
                if mv > 0:
                    bin_mask = remove_small_components(bin_mask, min_voxels=mv)
                # optional morphological closing to fill small holes
                bin_mask = binary_closing(bin_mask, structure=np.ones((3,3,3))).astype(np.uint8)
                f1s.append(f1_score(gts[i].ravel(), bin_mask.ravel(), zero_division=0))
            mean_f1 = float(np.mean(f1s))
            if mean_f1 > best[2]:
                best = (t, mv, mean_f1)
    return best  # (best_threshold, best_min_vox, best_score)

# ---------------- Evaluate and save per-case (uses tuned threshold/min_vox) ----------------
def evaluate_and_save_per_case_with_params(models, loader, df_subset, out_dir="./eval_outputs",
                                           save_nifti=True, threshold=0.5, min_vox=50):
    os.makedirs(out_dir, exist_ok=True)
    for m in models: m.eval()
    dice_metric = DiceMetric(include_background=True, reduction="mean")
    records = []
    with torch.no_grad():
        for idx, (img, msk) in enumerate(tqdm(loader, desc="Test Eval")):
            case_id = df_subset.iloc[idx]["case_id"] if "case_id" in df_subset.columns else f"case_{idx}"
            images = img.to(CONFIG["device"], dtype=torch.float32)
            masks = msk.to(CONFIG["device"], dtype=torch.float32)
            if images.ndim == 4:
                images = images.unsqueeze(0); masks = masks.unsqueeze(0)
            images, pad = pad_to_multiple_tensor(images, CONFIG["spatial_multiple"])
            masks, _ = pad_to_multiple_tensor(masks, CONFIG["spatial_multiple"])
            preds = []
            for model in models:
                p = torch.sigmoid(model(images))
                if p.shape != masks.shape:
                    p = F.interpolate(p, size=masks.shape[2:], mode="trilinear", align_corners=False)
                preds.append(p)
            ensemble = torch.mean(torch.stack(preds, dim=0), dim=0)
            pred_bin = (ensemble > threshold).float()
            gt_bin = (masks > 0.5).float()
            try:
                dice_metric(y_pred=pred_bin, y=gt_bin)
            except Exception:
                if pred_bin.shape != gt_bin.shape:
                    pred_bin = F.interpolate(pred_bin, size=gt_bin.shape[2:], mode="nearest")
                    dice_metric(y_pred=pred_bin, y=gt_bin)
                else:
                    raise
            pred_unp = unpad_tensor(pred_bin, pad)
            gt_unp = unpad_tensor(gt_bin, pad)
            pred_np = pred_unp.cpu().numpy()[0,0].astype(np.uint8)
            gt_np = gt_unp.cpu().numpy()[0,0].astype(np.uint8)
            if min_vox > 0:
                pred_np = remove_small_components(pred_np, min_voxels=min_vox)
            pred_np = binary_closing(pred_np, structure=np.ones((3,3,3))).astype(np.uint8)
            y_pred_flat = pred_np.ravel()
            y_true_flat = gt_np.ravel()
            if y_true_flat.sum() == 0 and y_pred_flat.sum() == 0:
                f1 = 1.0
            else:
                f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
            records.append({"case_id": case_id, "f1": float(f1)})
            if save_nifti:
                mask_path = df_subset.iloc[idx]["mask"]
                try:
                    ref_nii = nib.load(mask_path)
                    affine = ref_nii.affine
                except Exception:
                    affine = np.eye(4)
                out_path = os.path.join(out_dir, f"{case_id}_pred.nii.gz")
                nib.save(nib.Nifti1Image(pred_np.astype(np.uint8), affine), out_path)
    mean_dice = dice_metric.aggregate().item()
    dice_metric.reset()
    df_out = pd.DataFrame(records)
    df_out["dice"] = float(mean_dice)
    csv_path = os.path.join(out_dir, "per_case_metrics.csv")
    df_out.to_csv(csv_path, index=False)
    return float(mean_dice), float(df_out["f1"].mean()), csv_path

# ---------------- Main ----------------
if __name__ == "__main__":
    df = get_isles_dataframe(SEARCH_ROOT)
    if len(df) == 0:
        raise RuntimeError("No valid cases found. Check file naming and folder structure.")
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=CONFIG["seed"])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=CONFIG["seed"])

    train_ds = Stroke3DDataset(train_df.reset_index(drop=True), patch_size=CONFIG["patch_size"],
                               lesion_patch_prob=CONFIG["lesion_patch_prob"],
                               apply_n4=CONFIG["apply_n4_in_preprocessing"], augment=True,
                               preload_to_ram=CONFIG["preload_to_ram"], n_proc=max(1, cpu_count()-1))
    val_ds = Stroke3DDataset(val_df.reset_index(drop=True), patch_size=CONFIG["patch_size"],
                             lesion_patch_prob=0.0, apply_n4=CONFIG["apply_n4_in_preprocessing"],
                             augment=False, preload_to_ram=CONFIG["preload_to_ram"], n_proc=1)
    test_ds = Stroke3DDataset(test_df.reset_index(drop=True), patch_size=None,
                              lesion_patch_prob=0.0, apply_n4=CONFIG["apply_n4_in_preprocessing"],
                              augment=False, preload_to_ram=CONFIG["preload_to_ram"], n_proc=1)

    # Build WeightedRandomSampler to oversample positive cases
    # Use memory_cache (preloaded) to compute weights; fallback to reading cache files
    weights = []
    for i in range(len(train_ds)):
        if train_ds.memory_cache is not None:
            _, mask = train_ds.memory_cache[i]
        else:
            data = np.load(train_ds.cache_paths[i])
            mask = data["mask"]
        weights.append(5.0 if mask.sum() > 0 else 1.0)  # positive cases weight 5x
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], sampler=sampler,
                              num_workers=CONFIG["num_workers_train"], pin_memory=True,
                              persistent_workers=True, prefetch_factor=CONFIG["prefetch_factor"])
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                            num_workers=CONFIG["num_workers_val"], pin_memory=True,
                            persistent_workers=True, prefetch_factor=CONFIG["prefetch_factor"])
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                             num_workers=CONFIG["num_workers_val"], pin_memory=True,
                             persistent_workers=True, prefetch_factor=CONFIG["prefetch_factor"])

    models = get_models(CONFIG["device"])
    optimizers = [torch.optim.Adam(m.parameters(), lr=CONFIG["lr"]) for m in models]
    schedulers = [torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, CONFIG["epochs"])) for opt in optimizers]
    # Use FocalTversky combined with BCE for stability
    focal = FocalTversky(alpha=0.7, beta=0.3, gamma=0.75)
    bce = torch.nn.BCEWithLogitsLoss()
    def loss_fn(pred_logits, target):
        return 0.7 * focal(pred_logits, target) + 0.3 * bce(pred_logits, target)

    scaler = GradScaler()

    best_val_dice = -1.0
    best_epoch = -1

    # Sequential per-model training loop (one model at a time)
    for epoch in range(CONFIG["epochs"]):
        print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
        for i, model in enumerate(models):
            model.train()
            optimizer = optimizers[i]
            total_loss = 0.0
            n_batches = 0
            desc = f"Epoch {epoch+1} - Train model {i}"
            for images, masks in tqdm(train_loader, desc=desc, leave=False):
                images = images.to(CONFIG["device"], dtype=torch.float32)
                masks = masks.to(CONFIG["device"], dtype=torch.float32)
                if images.ndim == 4:
                    images = images.unsqueeze(0); masks = masks.unsqueeze(0)
                images, pad = pad_to_multiple_tensor(images, CONFIG["spatial_multiple"])
                masks, _ = pad_to_multiple_tensor(masks, CONFIG["spatial_multiple"])
                optimizer.zero_grad()
                with torch.amp.autocast(device_type="cuda" if CONFIG["device"].type == "cuda" else "cpu"):
                    output = model(images)
                    loss = loss_fn(output, masks)
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()
                n_batches += 1
            avg_loss = total_loss / max(1, n_batches)
            print(f"Model {i} train loss: {avg_loss:.4f}")

        # Validation (ensemble) every validate_every epochs
        if (epoch + 1) % CONFIG["validate_every"] == 0:
            val_dice, val_f1 = validate_ensemble(models, val_loader)
            print(f"Validation - Dice: {val_dice:.4f} | F1: {val_f1:.4f}")
            for sch in schedulers: sch.step()
            if val_dice > best_val_dice + 1e-4:
                best_val_dice = val_dice
                best_epoch = epoch + 1
                for i, m in enumerate(models):
                    torch.save(m.state_dict(), os.path.join(OUT_DIR, f"best_model_{i}.pth"))
                print(f"--- New Best Ensemble Saved (epoch {best_epoch}) ---")
            else:
                print("No improvement this validation step.")
        else:
            for sch in schedulers: sch.step()

    # Load best checkpoints if available
    for i, m in enumerate(models):
        ckpt = os.path.join(OUT_DIR, f"best_model_{i}.pth")
        if os.path.exists(ckpt):
            m.load_state_dict(torch.load(ckpt, map_location=CONFIG["device"]))
            print(f"Loaded checkpoint: {ckpt}")

    # Collect validation predictions and tune threshold/min_vox
    print("Collecting validation predictions for tuning...")
    preds_per_model, gts = collect_val_predictions(models, val_loader)
    print("Tuning threshold and min-component size on validation set...")
    best_thresh, best_minvox, best_score = tune_threshold_and_minvox(preds_per_model, gts,
                                                                      thresholds=np.linspace(0.3,0.7,21),
                                                                      min_vox_list=[10,25,50,100])
    print(f"Best threshold: {best_thresh:.3f}, best min_vox: {best_minvox}, val F1: {best_score:.4f}")

    # Final evaluation on test set using tuned params
    print("\nEvaluating best ensemble on test set with tuned params...")
    test_dice, test_f1, csv_path = evaluate_and_save_per_case_with_params(models, test_loader,
                                                                          test_df.reset_index(drop=True),
                                                                          out_dir="./eval_outputs",
                                                                          save_nifti=True,
                                                                          threshold=best_thresh,
                                                                          min_vox=best_minvox)
    print(f"Test Dice: {test_dice:.4f} | Test F1: {test_f1:.4f}")
    print(f"Per-case metrics saved to: {csv_path}")


Installing missing packages: ['monai']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 27.8 MB/s eta 0:00:00


2026-03-16 13:34:56.106225: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773668096.387569      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773668096.460641      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773668097.087347      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773668097.087405      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773668097.087408      55 computation_placer.cc:177] computation placer alr


Epoch 1/100


Epoch 1 - Train model 0:   0%|          | 0/175 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Model 0 train loss: 0.8358


Epoch 1 - Train model 1:  23%|██▎       | 40/175 [1:52:25<6:38:27, 177.09s/it]